# ADK による AI エージェント開発の基礎

このノートブックでは、ADK で会話型の AI エージェントを作成・利用する基本的な手順を確認します。

## 事前準備

**[ADB-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[ADB-02]**

インストールされたパッケージのバージョンを確認します。

In [2]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[ADB-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

**[ADB-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [2]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

## 初期設定

**[ADB-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

In [3]:
import os
from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import HTML, Markdown, display
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk.agents.llm_agent import LlmAgent
from google.adk.tools import google_search

agentplatform.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

## LlmAgent オブジェクトと AdkApp オブジェクトの作成

**[ADB-06]**

Grounding with Google Search を利用して、ユーザーの質問に回答する AI エージェント（LlmAgent オブジェクト）を作成します。

In [4]:
instruction = '''
あなたはユーザーの質問に回答するエージェントです。
- google_search を使用して、最新情報に基づいて回答してください。
- フレンドリーな会話を心がけてください。
'''

search_agent = LlmAgent(
    name='search_agent',
    model='gemini-3.5-flash-lite',
    description='Google 検索を用いて質問に回答するエージェント',
    instruction=instruction,
    tools=[google_search],
)


**[ADB-07]**

作成した LlmAgent オブジェクトを含む AdkApp オブジェクトを作成します。

このオブジェクトのメソッドを通じて、AI エージェントと対話します。

In [5]:
search_agent_app = AdkApp(
    agent=search_agent,
    app_name='search_agent_app',
)

## AdkApp オブジェクトと会話するアプリケーションの作成

**[ADB-08]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

In [6]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = session['id']

        result = []
        events = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            events.append(event)
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
        return '\n'.join(result), events

## 会話に伴うイベントデータの確認

**[ADB-09]**

先ほど作成した AdkApp オブジェクト `search_agent_app` を使用する ChatClient オブジェクトを作成して、ユーザーのメッセージを送信します。

変数 `response` に AI エージェントの応答メッセージがマークダウンテキストで格納されます。

In [7]:
chat_client = ChatClient(search_agent_app)

query = '''
高田馬場のおすすめのカレー屋は？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

/root/.local/lib/python3.13/site-packages/agentplatform/frameworks/adk.py:1146: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/root/.local/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


高田馬場は、実はバラエティ豊かなカレーの名店が集まる激戦区の一つです！その中でも特に評判が高く、おすすめのお店をいくつか厳選してご紹介しますね。

### 1. カレー専門店 ブラザー
* **特徴:** 高田馬場を代表する大人気スパイスカレー店。名物の「鯖キーマ」や、旨味が凝縮された欧風・スパイスの融合系カレーが楽しめます。
* **ここがおすすめ:** 定番のチキンカレーなども非常にクオリティが高く、スパイスの芳醇な香りと深みのある辛さがクセになります。行列ができることも多い実力派です。

### 2. プネウマカレー
* **特徴:** 伊吹島産のアンチョビを隠し味に使った、スパイス感たっぷりの個性派チキンカレーのお店。
* **ここがおすすめ:** なんといってもその「高コスパ」が魅力で、ボリュームのある美味しいチキンカレーを手頃な価格でサッと楽しめます。スピーディーに出てくるので、お忙しい方にもぴったりです。

### 3. カリーライス専門店 エチオピア 高田馬場店
* **特徴:** 神保町に本店を構える、薬膳スパイスたっぷりの名店「エチオピア」の支店。
* **ここがおすすめ:** サラッとしたスパイスの効いたルーと、ホクホクの「豆カレー」などが大人気。辛さを選べるので、本格的なスパイスカレーを自分の好みの辛さで楽しみたいときにおすすめです。

### 4. カレーハウス 横浜ボンベイ 高田馬場店
* **特徴:** 伝説の「デリー」の流れをくむ、横浜ボンベイの味が楽しめるお店。
* **ここがおすすめ:** 深いコクとシャープな辛さが特徴の「ボンベイカレー」や、カシミールカレーなどの辛口メニューが充実しています。ガツンとスパイスと辛さを堪能したいときにぜひ。

---

気分に合わせて、王道のスパイスカレーなら「ブラザー」、サクッとコスパよく食べるなら「プネウマカレー」、サラリと薬膳系なら「エチオピア」など、使い分けてみるのも楽しいですよ！
気になるお店はありましたか？

**[ADB-10]**

変数 `events` には、AI エージェントの処理過程の情報を含むさまざまなイベントデータが格納されています。

>ファンクションコールなど、多段階のループ処理を行った場合は、複数のイベントがリスト形式で格納されますが、今回の場合は、単一のイベント `evetns[0]` のみが含まれます。

ディクショナリ形式のイベントデータに含まれる Key を確認します。

In [8]:
events[0].keys()

dict_keys(['model_version', 'content', 'grounding_metadata', 'finish_reason', 'usage_metadata', 'invocation_id', 'author', 'actions', 'node_info', 'id', 'timestamp'])

**[ADB-11]**

特に `author` と `timestamp` には、イベントを発行した LlmAgent オブジェクトの名前と、タイムスタンプが格納されています。

In [9]:
author = events[0]['author']
timestamp = events[0]['timestamp']
print(f'{author}: {datetime.fromtimestamp(timestamp, tz=ZoneInfo('Asia/Tokyo'))}')

search_agent: 2026-09-01 08:38:06.987207+09:00


**[ADB-12]**

`content` には、応答メッセージが含まれます。

In [10]:
events[0]['content']

{'parts': [{'text': '高田馬場は、実はバラエティ豊かなカレーの名店が集まる激戦区の一つです！その中でも特に評判が高く、おすすめのお店をいくつか厳選してご紹介しますね。\n\n### 1. カレー専門店 ブラザー\n* **特徴:** 高田馬場を代表する大人気スパイスカレー店。名物の「鯖キーマ」や、旨味が凝縮された欧風・スパイスの融合系カレーが楽しめます。\n* **ここがおすすめ:** 定番のチキンカレーなども非常にクオリティが高く、スパイスの芳醇な香りと深みのある辛さがクセになります。行列ができることも多い実力派です。\n\n### 2. プネウマカレー\n* **特徴:** 伊吹島産のアンチョビを隠し味に使った、スパイス感たっぷりの個性派チキンカレーのお店。\n* **ここがおすすめ:** なんといってもその「高コスパ」が魅力で、ボリュームのある美味しいチキンカレーを手頃な価格でサッと楽しめます。スピーディーに出てくるので、お忙しい方にもぴったりです。\n\n### 3. カリーライス専門店 エチオピア 高田馬場店\n* **特徴:** 神保町に本店を構える、薬膳スパイスたっぷりの名店「エチオピア」の支店。\n* **ここがおすすめ:** サラッとしたスパイスの効いたルーと、ホクホクの「豆カレー」などが大人気。辛さを選べるので、本格的なスパイスカレーを自分の好みの辛さで楽しみたいときにおすすめです。\n\n### 4. カレーハウス 横浜ボンベイ 高田馬場店\n* **特徴:** 伝説の「デリー」の流れをくむ、横浜ボンベイの味が楽しめるお店。\n* **ここがおすすめ:** 深いコクとシャープな辛さが特徴の「ボンベイカレー」や、カシミールカレーなどの辛口メニューが充実しています。ガツンとスパイスと辛さを堪能したいときにぜひ。\n\n---\n\n気分に合わせて、王道のスパイスカレーなら「ブラザー」、サクッとコスパよく食べるなら「プネウマカレー」、サラリと薬膳系なら「エチオピア」など、使い分けてみるのも楽しいですよ！\n気になるお店はありましたか？',
   'thought_signature': 'AY89a18cn9JFIJZSDinjwQqaLZlavJD2_-vxlvuUKAaqHzNlHSh_Z--O3rkoKqfgvPSOL2Vq

**[ADB-13]**

Grounding with Google Search を使用した場合は、Google 検索に使用したキーワードも確認できます。

In [11]:
events[0]['grounding_metadata']['web_search_queries']

['高田馬場 おすすめ カレー', '高田馬場 カレー 人気店']

**[ADB-14]**

同じキーワードで検索を実行するボタンを表示する HTML テキストも用意されます。

In [12]:
display(HTML(
    events[0]['grounding_metadata']['search_entry_point']['rendered_content']
))

**[ADB-15]**

これまでの会話履歴は、AdkApp オブジェクト内の SessionService オブジェクトに保存されているので、そのまま会話を継続できます。

In [13]:
query = '''
特に家族連れにおすすめなのは？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

家族連れ（小さなお子様連れ）で高田馬場周辺のカレー屋さんに行く場合は、先ほど挙げたような「マニアックなスパイス専門店」だと席が狭かったり、子どもには辛すぎたりすることがあります。

そのため、家族連れで行くなら**「インド・ネパール系のレストラン（インネパ店）」**や**「広めの席があるお店・商業施設内のお店」**を選ぶのがおすすめです！高田馬場でおすすめのスポットはこちらです。

### 1. 駅周辺にあるインド・ネパール料理店（例：ナマステヒマール など）
* **おすすめの理由:** 高田馬場は学生街ということもあり、ナンとカレーが食べられるインド・ネパール料理店が非常にたくさんあります。
* **子連れに嬉しいポイント:** 
  * 辛くない「バターチキンカレー」や「チーズナン」など、子どもが大好きなメニューが必ずと言っていいほどあります。
  * 一般的にテーブル席が広く、子ども用のイスや食器が用意されているお店が多いです。
  * 店員さんも子どもウェルカムな雰囲気のところが多く、安心して利用できます。

### 2. デリー系や欧風カレー、チェーン店（ボックス席があるお店など）
* **おすすめの理由:** カレー自体の味がしっかりしていて大人も満足できつつ、辛さの調節（甘口など）ができるお店。
* また、高田馬場駅周辺にはファミレスや、ゆったり座れるチェーンのカレー店（CoCo壱番屋など）もあるため、「子供が食べ慣れた味がいい」「周りを気にせずゆっくり座りたい」という場合は、こうした選択肢も安心です。

---

**💡 ご家族で快適に過ごすためのコツ**
もし本格的なスパイスカレー店（「ブラザー」など）に家族で行きたい場合は、**席数が少なく行列ができやすい（カウンター席メインであることが多い）**ため、小さなお子様連れだと少しハードルが高くなってしまいます。

「子どもも一緒に美味しくナンや甘口カレーをワイワイ楽しみたい！」ということであれば、駅周辺の**インド料理店**を狙うのが一番間違いありませんよ！

**[ADB-16]**

今回の応答メッセージの生成に使用した検索キーワードを確認します。

In [14]:
events[0]['grounding_metadata']['web_search_queries']

['高田馬場 カレー 家族連れ 子連れ']